# SAC vs FastSAC Comparison on HalfCheetah-v5

Side-by-side training runs logged to **W&B** under the same group for easy comparison.

| Run | Algorithm | feature_update_ratio | actor_update_freq | target_update_freq | Expected net updates |
|-----|-----------|---------------------|-------------------|--------------------|---------------------|
| A   | SAC       | N/A (UTD=1)          | 1                 | 1                  | ~96K                |
| B   | FastSAC   | 1000 (= FPB)         | 1                 | 1                  | ~96K (parity)       |
| C   | FastSAC   | 2000 (= 2×FPB)       | 1                 | 1                  | ~192K (2× updates)  |
| D   | FastSAC   | 1000 (= FPB)         | 2                 | 1                  | ~96K (actor: ~48K)  |

Reference: `spectral-rl/spectralrl/algo/state/sac/agent.py`

In [1]:
from __future__ import annotations

import gymnasium as gym
import torch

from rlopt.agent.sac import SAC, SACRLOptConfig
from rlopt.agent.sac.fast_sac import FastSAC, FastSACConfig, FastSACRLOptConfig
from rlopt.config_base import NetworkConfig
from rlopt.env_utils import make_parallel_env

/Users/donaldheddesheimer/miniforge3/envs/RL/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

Shared hyper-parameters for all runs. Adjust `TOTAL_FRAMES` for longer training.

In [2]:
# ---- Shared experiment settings ----
ENV_NAME = "HalfCheetah-v5"
SEED = 42
TOTAL_FRAMES = 100_000       # increase for real experiments (e.g. 1_000_000)
FRAMES_PER_BATCH = 1_000
INIT_RANDOM_FRAMES = 5_000
BATCH_SIZE = 256
BUFFER_SIZE = 1_000_000
LR = 3e-4
NUM_ENVS = 1
DEVICE = "cpu"               # or "cuda" / "mps"

# W&B settings
WANDB_PROJECT = "RLOpt"
WANDB_GROUP = "sac_vs_fastsac"  # groups runs together on dashboard

# Resolve obs/action dims from the env
dummy_env = gym.make(ENV_NAME)
OBS_DIM = dummy_env.observation_space.shape[0]
ACT_DIM = dummy_env.action_space.shape[0]
dummy_env.close()
print(f"{ENV_NAME}: obs_dim={OBS_DIM}, action_dim={ACT_DIM}")

HalfCheetah-v5: obs_dim=17, action_dim=6


In [3]:
def make_sac_config(exp_name: str = "SAC") -> SACRLOptConfig:
    """Baseline SAC config."""
    cfg = SACRLOptConfig()
    cfg.seed = SEED
    cfg.device = DEVICE

    cfg.env.env_name = ENV_NAME
    cfg.env.num_envs = NUM_ENVS

    cfg.collector.frames_per_batch = FRAMES_PER_BATCH
    cfg.collector.total_frames = TOTAL_FRAMES
    cfg.collector.init_random_frames = INIT_RANDOM_FRAMES

    cfg.loss.mini_batch_size = BATCH_SIZE
    cfg.replay_buffer.size = BUFFER_SIZE
    cfg.optim.lr = LR
    cfg.optim.target_update_polyak = 0.995
    cfg.sac.utd_ratio = 1.0

    cfg.policy.input_dim = OBS_DIM
    cfg.policy.num_cells = [256, 256]
    cfg.policy.activation_fn = "relu"

    cfg.q_function = NetworkConfig(
        num_cells=[256, 256],
        input_dim=OBS_DIM + ACT_DIM,
        input_keys=["observation", "action"],
        activation_fn="relu",
    )

    # W&B logging
    cfg.logger.backend = "wandb"
    cfg.logger.project_name = WANDB_PROJECT
    cfg.logger.group_name = WANDB_GROUP
    cfg.logger.exp_name = exp_name

    return cfg


def make_fastsac_config(
    exp_name: str = "FastSAC",
    feature_update_ratio: int = 1,
    actor_update_freq: int = 1,
    target_update_freq: int = 1,
) -> FastSACRLOptConfig:
    """FastSAC config with scheduling knobs."""
    cfg = FastSACRLOptConfig()
    cfg.seed = SEED
    cfg.device = DEVICE

    cfg.env.env_name = ENV_NAME
    cfg.env.num_envs = NUM_ENVS

    cfg.collector.frames_per_batch = FRAMES_PER_BATCH
    cfg.collector.total_frames = TOTAL_FRAMES
    cfg.collector.init_random_frames = INIT_RANDOM_FRAMES

    cfg.loss.mini_batch_size = BATCH_SIZE
    cfg.replay_buffer.size = BUFFER_SIZE
    cfg.optim.lr = LR
    cfg.optim.target_update_polyak = 0.995

    # FastSAC scheduling
    cfg.sac.feature_update_ratio = feature_update_ratio
    cfg.sac.actor_update_freq = actor_update_freq
    cfg.sac.target_update_freq = target_update_freq

    cfg.policy.input_dim = OBS_DIM
    cfg.policy.num_cells = [256, 256]
    cfg.policy.activation_fn = "relu"

    cfg.q_function = NetworkConfig(
        num_cells=[256, 256],
        input_dim=OBS_DIM + ACT_DIM,
        input_keys=["observation", "action"],
        activation_fn="relu",
    )

    # W&B logging
    cfg.logger.backend = "wandb"
    cfg.logger.project_name = WANDB_PROJECT
    cfg.logger.group_name = WANDB_GROUP
    cfg.logger.exp_name = exp_name

    return cfg

## Define comparison runs

Each entry: `(label, config, agent_class)`

In [4]:
# feature_update_ratio = FRAMES_PER_BATCH gives parity with SAC (UTD=1)
FPB = FRAMES_PER_BATCH

RUNS = [
    ("SAC-baseline",        make_sac_config("SAC-baseline"),                                                                    SAC),
    ("FastSAC-1x",          make_fastsac_config("FastSAC-1x",          feature_update_ratio=FPB,   actor_update_freq=1),        FastSAC),
    ("FastSAC-2x",          make_fastsac_config("FastSAC-2x",          feature_update_ratio=FPB*2, actor_update_freq=1),        FastSAC),
    ("FastSAC-delayed-act", make_fastsac_config("FastSAC-delayed-act", feature_update_ratio=FPB,   actor_update_freq=2),        FastSAC),
]

print(f"{len(RUNS)} runs queued — all logged to W&B project '{WANDB_PROJECT}', group '{WANDB_GROUP}'")
for name, cfg, cls in RUNS:
    fur = getattr(cfg.sac, 'feature_update_ratio', None)
    auf = getattr(cfg.sac, 'actor_update_freq', 1)
    tuf = getattr(cfg.sac, 'target_update_freq', 1)
    print(f"  {name:25s}  cls={cls.__name__:8s}  FUR={fur}  AUF={auf}  TUF={tuf}")

4 runs queued — all logged to W&B project 'RLOpt', group 'sac_vs_fastsac'
  SAC-baseline               cls=SAC       FUR=None  AUF=1  TUF=1
  FastSAC-1x                 cls=FastSAC   FUR=1000  AUF=1  TUF=1
  FastSAC-2x                 cls=FastSAC   FUR=2000  AUF=1  TUF=1
  FastSAC-delayed-act        cls=FastSAC   FUR=1000  AUF=2  TUF=1


## Train all runs sequentially

Each run creates a separate W&B run with the experiment name as the run label.  
After training finishes you can compare them on the W&B dashboard by filtering the **group** `sac_vs_fastsac`.

In [5]:
results = {}

for name, cfg, cls in RUNS:
    print(f"\n{'='*60}")
    print(f"Starting: {name}  ({cls.__name__})")
    print(f"{'='*60}")

    env = make_parallel_env(cfg)
    agent = cls(env, cfg, logger=None)
    agent.train()

    info = {
        "total_network_updates": agent.total_network_updates,
        "total_actor_updates": getattr(agent, "total_actor_updates", agent.total_network_updates),
        "total_target_updates": getattr(agent, "total_target_updates", agent.total_network_updates),
        "alpha": agent.loss_module.log_alpha.exp().item(),
    }
    results[name] = info
    print(f"  Done — net_updates={info['total_network_updates']}, "
          f"actor_updates={info['total_actor_updates']}, "
          f"alpha={info['alpha']:.4f}")


Starting: SAC-baseline  (SAC)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/donaldheddesheimer/.netrc.
wandb: Currently logged in as: dheddesheimer3 (dheddesheimer3-georgia-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


100%|██████████| 100000/100000 [12:10<00:00, 136.95it/s, ep_ret=1378.7, ep_len=1000]


  Done — net_updates=96000, actor_updates=96000, alpha=0.2114

Starting: FastSAC-1x  (FastSAC)


episode/length,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
episode/return,▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▃▃▃▄▄▄▄▄▅▅▅▆▆▆▇▇▇▇▇███
time/collect,▁▁▃▄▆▇▇▇▇▇▇▇▇████████████████▇▇▇▆▆▆▆▆▅▅▅
time/rb - sample,▃▁▆▇█▇▆▅▆▇▆▆▅▆▆▅▆▅▅▅▄▄▄▄▄▄▄▅▅▅▆▅▄▃▃▂▂▂▂▁
time/replay_extend,█▆▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
time/speed,▆██▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
time/train,▁▁▅▅▇▇▇▇▇▇▇█████████████████████████████
time/update,█▄▅▅▅▄▄▄▄▄▄▄▃▄▄▄▃▃▃▃▃▃▃▃▃▃▃▃▄▃▃▃▃▂▂▂▂▂▁▁
train/alpha,█▆▄▃▂▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▃▃▃▃
train/loss_actor,█▇▇▇▇▇▇█████████████▇▇▇▇▇▇▆▆▆▆▅▅▅▅▄▃▃▂▂▁
+7,...


...

episode/length,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
episode/return,▁▁▁▁▁▁▁▂▂▂▂▂▂▃▃▃▄▄▄▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇█████
time/collect,▁▂▂▃▃▅▅▅▅▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇██████████
time/rb - sample,▁▁▁▇▇▇▇▇████████████████████████████████
time/replay_extend,▄█▇▆▅▅▃▃▃▃▁▂▂▃▂▂▁▂▂▂▂▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
time/speed,▇█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
time/train,▁▁▁▁▄▅▅▆▆▇▇▇▇▇▇▇████████████████████████
time/update,▁▁▁█████████████████████████████████████
train/alpha,█▂▁▁▁▁▁▁▁▂▂▂▃▃▄▄▄▄▅▅▅▅▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇█
train/loss_actor,███████████▇▇▇▇▆▆▆▅▅▅▅▅▄▄▄▃▃▃▃▃▃▃▃▂▂▂▁▁▁
+7,...


  0%|          | 0/100000 [00:00<?, ?it/s]

2026-03-11 08:31:43,202 [torchrl][INFO]    collect took 472.1719 msec (total =  47.2172 sec since last reset). [END]
2026-03-11 08:31:43,202 [torchrl][INFO]    rb - sample took 0.2871 msec (total =  55.1288 sec since last reset). [END]
2026-03-11 08:31:43,202 [torchrl][INFO]    replay_extend took 0.5038 msec (total =  0.0504 sec since last reset). [END]
2026-03-11 08:31:43,203 [torchrl][INFO]    train took 12875.0203 msec (total =  1287.5020 sec since last reset). [END]
2026-03-11 08:31:43,203 [torchrl][INFO]    update took 6.4043 msec (total =  1229.6166 sec since last reset). [END]


100%|██████████| 100000/100000 [10:00<00:00, 166.39it/s, ep_ret=1642.9, ep_len=1000]

  Done — net_updates=96000, actor_updates=48000, alpha=0.2339


## Summary table

In [6]:
header = f"{'Run':25s} {'Net Upd':>10s} {'Actor Upd':>10s} {'Target Upd':>10s} {'Alpha':>8s}"
print(header)
print("-" * len(header))
for name, info in results.items():
    print(f"{name:25s} {info['total_network_updates']:>10d} "
          f"{info['total_actor_updates']:>10d} "
          f"{info['total_target_updates']:>10d} "
          f"{info['alpha']:>8.4f}")

Run                          Net Upd  Actor Upd Target Upd    Alpha
-------------------------------------------------------------------
SAC-baseline                   96000      96000      96000   0.2114
FastSAC-1x                     96000      96000      96000   0.2114
FastSAC-2x                    192000     192000     192000   0.2902
FastSAC-delayed-act            96000      48000      96000   0.2339
